In [1]:
import os
import json
import re
import pandas as pd
from tqdm import tqdm
from pycocotools.coco import COCO

ou sauvegarder les resultat

In [17]:
import os

folder = "../data/processed/captions"

os.makedirs(folder, exist_ok=True)

print(os.listdir(folder))

['caption_audit.json', 'kb_captions_clean.csv']


Knowledge Base IDs(splits)

In [ ]:
with open("../splits/splits_all.json", "r") as f:
    splits = json.load(f)
#id's of omgs in kb 
kb_ids = splits["kb"]

print("Nombre d'images dans KB :", len(kb_ids))

Nombre d'images dans KB : 10000


Load the Original Caption 

In [3]:
coco_caps = COCO("../data/annotations/captions_train2017.json")

loading annotations into memory...
Done (t=0.75s)
creating index...
index created!


Captions for KB Images

In [4]:
caption_rows = []

for iid in tqdm(kb_ids):
    ann_ids = coco_caps.getAnnIds(imgIds=[iid])
    anns = coco_caps.loadAnns(ann_ids)

    for ann in anns:
        caption_rows.append({
            "image_id": iid,
            "caption_raw": ann["caption"]
        })

df_captions = pd.DataFrame(caption_rows)

100%|██████████| 10000/10000 [00:02<00:00, 4679.89it/s]


clean funtion

In [5]:
def clean_caption(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text.strip()

# 5. Créer une version nettoyée
df_captions["caption_clean"] = (
    df_captions["caption_raw"].apply(clean_caption)
)

In [5]:
random.seed(42)
np.random.seed(42)

verifications

In [ ]:
print("\nNombre total de captions :", len(df_captions))
print("Captions originales manquantes :", df_captions["caption_raw"].isna().sum())
print("Captions nettoyées manquantes :", df_captions["caption_clean"].isna().sum())
print("Doublons exacts :", df_captions.duplicated().sum())


Nombre total de captions : 50024
Captions originales manquantes : 0
Captions nettoyées manquantes : 0
Doublons exacts : 13


In [7]:
display(df_captions.head(10))

,image_id,caption_raw,caption_clean
0,101014,A table filled with some very nice looking food.,a table filled with some very nice looking food
1,101014,A black tabletop with two dishes of food and t...,a black tabletop with two dishes of food and t...
2,101014,A couple of plates filled with different foods.,a couple of plates filled with different foods
3,101014,Two plates of food on a table at a restaurant.,two plates of food on a table at a restaurant
4,101014,Two plates of well prepared food sitting next ...,two plates of well prepared food sitting next ...
5,203463,An orange-and-black cat wearing a tiny black hat.,an orangeandblack cat wearing a tiny black hat
6,203463,a small calico cat on a pink blanket,a small calico cat on a pink blanket
7,203463,A calico cat wearing a cowboy hat on its head.,a calico cat wearing a cowboy hat on its head
8,203463,A cat sitting on top of a persons lap.,a cat sitting on top of a persons lap
9,203463,"A brown, black, and white cat that is wearing ...",a brown black and white cat that is wearing a ...


save the results

In [8]:
output_folder = "../data/processed/captions"
os.makedirs(output_folder, exist_ok=True)

df_captions.to_csv(
    f"{output_folder}/kb_captions_clean.csv",
    index=False
)

print("\nFichier sauvegardé :", os.listdir(output_folder))


Fichier sauvegardé : ['kb_captions_clean.csv']


In [9]:
# Nombre de captions par image
captions_per_image = df_captions.groupby("image_id").size()

# Images avec un nombre différent de 5 captions
unusual_images = captions_per_image[captions_per_image != 5]

print("Nombre d'images analysées :", len(captions_per_image))
print("Images avec un nombre différent de 5 captions :", len(unusual_images))

display(unusual_images)

Nombre d'images analysées : 10000
Images avec un nombre différent de 5 captions : 23


image_id
34938     6
50031     6
52109     7
62943     6
80739     6
82273     6
85777     6
100008    6
162256    6
183646    6
341894    6
358884    6
386864    6
401720    6
404148    6
449768    6
464498    6
479332    6
484277    6
487036    6
492215    6
522013    6
525318    6
dtype: int64

In [10]:
duplicates = df_captions[
    df_captions.duplicated(
        subset=["image_id", "caption_raw"],
        keep=False
    )
]

print("Nombre de lignes concernées par les doublons :",
      len(duplicates))

display(duplicates.head(20))

Nombre de lignes concernées par les doublons : 26


,image_id,caption_raw,caption_clean
9757,52109,Three teenagers are playing tennis on a court ...,three teenagers are playing tennis on a court ...
9759,52109,Three teenagers are playing tennis on a court ...,three teenagers are playing tennis on a court ...
12765,43893,A man standing in front of a TV in a bedroom.,a man standing in front of a tv in a bedroom
12766,43893,A man standing in front of a TV in a bedroom.,a man standing in front of a tv in a bedroom
15012,404618,A yellow and black fire hydrant sitting on a s...,a yellow and black fire hydrant sitting on a s...
15013,404618,A yellow and black fire hydrant sitting on a s...,a yellow and black fire hydrant sitting on a s...
18194,431756,A black and white cat sitting on a chair.,a black and white cat sitting on a chair
18195,431756,A black and white cat sitting on a chair.,a black and white cat sitting on a chair
19912,516725,A woman standing in a bathroom holding a phone.,a woman standing in a bathroom holding a phone
19915,516725,A woman standing in a bathroom holding a phone.,a woman standing in a bathroom holding a phone


In [11]:
print("Nombre total d'IDs :", len(kb_ids))
print("Nombre d'IDs uniques :", len(set(kb_ids)))
print("IDs répétés :", len(kb_ids) - len(set(kb_ids)))

Nombre total d'IDs : 10000
Nombre d'IDs uniques : 10000
IDs répétés : 0


verification of statistics

In [14]:
print("Nombre d'images :", df_captions["image_id"].nunique())
print("Nombre total de captions :", len(df_captions))

print("\nDistribution du nombre de captions par image :")
print(df_captions.groupby("image_id").size().value_counts().sort_index())

print("\nNombre de captions répétées pour une même image :",
      df_captions.duplicated(
          subset=["image_id", "caption_raw"]
      ).sum())

Nombre d'images : 10000
Nombre total de captions : 50024

Distribution du nombre de captions par image :
5    9977
6      22
7       1
Name: count, dtype: int64

Nombre de captions répétées pour une même image : 13


save the results

In [18]:
df_captions.to_csv(
    f"{folder}/kb_captions_clean.csv",
    index=False
)

In [16]:
audit = {
    "number_of_images": int(df_captions["image_id"].nunique()),
    "number_of_captions": int(len(df_captions)),
    "missing_raw_captions": int(df_captions["caption_raw"].isna().sum()),
    "missing_clean_captions": int(df_captions["caption_clean"].isna().sum()),
    "duplicate_captions": int(
        df_captions.duplicated(
            subset=["image_id", "caption_raw"]
        ).sum()
    )
}

with open("../data/processed/captions/caption_audit.json", "w") as f:
    json.dump(audit, f, indent=4)

print("Caption audit saved successfully!")

Caption audit saved successfully!
